# Capstone Project — Build and Harden a Production Claude System

**Level: Intermediate**

## Objective

Integrate everything from Phases 1-5 into one production-grade Claude system, mirroring the CCAR-F exam's 6 scenarios and 4 Preparation Exercises. You will assemble a multi-tool support agent with escalation logic (Exercise 1 / Scenario 1), a Claude Code team configuration (Exercise 2 / Scenarios 2,4,5), a structured data extraction pipeline with validation-retry and batch processing (Exercise 3 / Scenario 6), and a multi-agent research pipeline with provenance tracking (Exercise 4 / Scenario 3). Each step cites the lessons and task statements it exercises.

### What You'll Build

A single repository that demonstrates production judgment across all five domains:

1. **Support Resolution Agent** - agentic loop, verify-before-refund gate, interception hook, structured MCP errors, and calibrated escalation.
2. **Claude Code Team Config** - CLAUDE.md hierarchy, path-specific rules, a forked skill, a project slash command, `.mcp.json` with env expansion, and a CI review command.
3. **Extraction Pipeline** - tool_use + JSON schema (nullable/enum), validation-retry with error feedback, batch submission with custom_id, and confidence-based human review routing.
4. **Research Pipeline** - coordinator/subagent orchestration, parallel Task calls, structured error propagation, and claim-source provenance with conflict/temporal handling.

### Rules

- Ground every design choice in the exam anti-patterns: no NL loop termination, no prompt-only enforcement where determinism is required, no self-review in place of independent review, no generic error statuses, no arbitrary selection among conflicting sources.
- Guard every live API call so the notebook runs end-to-end without ANTHROPIC_API_KEY.
- Prefer deterministic mechanisms (gates, hooks, schemas, validation) over prompt-only guidance.
- Cite the lesson/task-statement number in each step's rationale.

## Prerequisites

Before starting, make sure you have:

- [ ] Completed Phases 1-5 (all 29 lessons)
- [ ] Python 3 kernel with the `anthropic` SDK installed
- [ ] Optional: `ANTHROPIC_API_KEY` and the Claude Code CLI for the live/CLI cells
- [ ] Optional: a GITHUB_TOKEN/DB_URL in the environment for MCP env-expansion demos

**Estimated time:** 4-6 hours

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Step 1: Support Agent - Agentic Loop + Verify-Before-Refund Gate

**Lesson reference:** Lessons 1.1, 1.4, 1.5 (Task Statements 1.1, 1.4, 1.5); Scenario 1 / Exercise 1

Build the core support agent. Implement a `stop_reason`-driven loop and a programmatic prerequisite gate so `process_refund` is blocked until `get_customer` returns a verified ID (Sample Question 1 - option A). Add a PostToolUse normalizer and a pre-call interception hook that blocks refunds over $500 and redirects to escalation.

In [ ]:
class Session:
    def __init__(self):
        self.verified_customer_id = None

REFUND_LIMIT = 500

def get_customer(session, identifier):
    session.verified_customer_id = "CUST-8842"
    return session.verified_customer_id

def pre_tool_use_hook(session, tool, args):
    if tool == "process_refund":
        if not session.verified_customer_id:
            return {"action": "block", "redirect": "get_customer",
                    "reason": "identity not verified"}
        if args.get("amount", 0) > REFUND_LIMIT:
            return {"action": "block", "redirect": "escalate_to_human",
                    "reason": f"over ${REFUND_LIMIT}"}
    return {"action": "allow"}

def process_refund(session, amount):
    d = pre_tool_use_hook(session, "process_refund", {"amount": amount})
    if d["action"] == "block":
        return {"blocked": True, **d}
    return {"blocked": False, "result": f"Refunded ${amount} to {session.verified_customer_id}"}

s = Session()
print(process_refund(s, 50))          # blocked: unverified
get_customer(s, "dana@example.com")
print(process_refund(s, 750))         # blocked: over limit -> escalate
print(process_refund(s, 50))          # success

### Checkpoint 1: Verify Support Agent - Agentic Loop + Verify-Before-Refund Gate

Confirm the gate and hook behave deterministically across the three cases.

In [ ]:
s2 = Session()
assert process_refund(s2, 50)["blocked"]           # unverified blocked
get_customer(s2, 'x')
assert process_refund(s2, 750)["redirect"] == "escalate_to_human"
assert not process_refund(s2, 50)["blocked"]       # verified + under limit
print("Checkpoint 1 PASSED: deterministic gate + interception hook")

## Step 2: Support Agent - Structured MCP Errors + Calibrated Escalation

**Lesson reference:** Lessons 2.2, 5.2 (Task Statements 2.2, 5.2); Scenario 1 / Sample Questions 1-3

Give tools structured errors (errorCategory, isRetryable) and add escalation criteria. Honor explicit human requests immediately, escalate on policy gaps, and ask for more identifiers on multiple matches. Reject sentiment/self-confidence as triggers.

In [ ]:
def mcp_error(category, message, retryable, customer_msg=None):
    return {"isError": True, "errorCategory": category, "isRetryable": retryable,
            "description": message, "customerMessage": customer_msg}

def should_escalate(ctx):
    if ctx.get("explicit_human_request"):
        return (True, "customer asked for a human")
    if ctx.get("policy_gap"):
        return (True, "policy ambiguous/silent")
    if ctx.get("no_progress"):
        return (True, "no meaningful progress")
    return (False, "resolve autonomously")

def resolve_customer(matches):
    if len(matches) != 1:
        return {"action": "ask_identifier"}
    return {"action": "proceed", "customer": matches[0]}

print(mcp_error("business", "Refund over limit", False, "A specialist will help."))
print(should_escalate({"explicit_human_request": True}))
print(should_escalate({"sentiment": "angry"}))       # not a trigger
print(resolve_customer([{"id": "A"}, {"id": "B"}]))

### Checkpoint 2: Verify Support Agent - Structured MCP Errors + Calibrated Escalation

Verify escalation honors the exam's triggers and ignores sentiment.

In [ ]:
assert should_escalate({"explicit_human_request": True})[0] is True
assert should_escalate({"sentiment": "angry"})[0] is False
assert resolve_customer([{"id": "A"}, {"id": "B"}])["action"] == "ask_identifier"
print("Checkpoint 2 PASSED: structured errors + calibrated escalation")

## Step 3: Claude Code Team Configuration

**Lesson reference:** Lessons 3.1-3.3, 3.6, 2.4 (Task Statements 3.1-3.3, 3.6, 2.4); Scenarios 2,4,5 / Exercise 2

Scaffold the team config: a project `CLAUDE.md` with `@import`, path-specific `.claude/rules/` with glob frontmatter, a forked tool-restricted skill, a project `/review` command, a `.mcp.json` with `${ENV}` expansion, and a CI review command using `-p --output-format json`.

In [ ]:
import os, json, pathlib
PROJ = pathlib.Path('capstone_project'); PROJ.mkdir(exist_ok=True)
os.chdir(PROJ)
for d in ['.claude/rules', '.claude/commands', '.claude/skills/analyze']:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

pathlib.Path('CLAUDE.md').write_text(
    '# Orders Service\n## Standards\n- Python 3.12, type hints\n'
    '@import .claude/rules/testing.md\n', encoding='utf-8')
pathlib.Path('.claude/rules/testing.md').write_text(
    '---\npaths: ["**/*.test.*"]\n---\n# Tests\n- pytest; AAA\n', encoding='utf-8')
pathlib.Path('.claude/commands/review.md').write_text(
    '---\ndescription: Team review checklist\nargument-hint: <path>\n---\n'
    'Review $ARGUMENTS: report bugs/security; skip minor style.\n', encoding='utf-8')
pathlib.Path('.claude/skills/analyze/SKILL.md').write_text(
    '---\nname: analyze\ncontext: fork\nallowed-tools: Read, Grep, Glob\n'
    'argument-hint: <package>\n---\nMap structure of $ARGUMENTS.\n', encoding='utf-8')
pathlib.Path('.mcp.json').write_text(json.dumps({
    'mcpServers': {'github': {'command': 'npx',
        'args': ['-y', '@modelcontextprotocol/server-github'],
        'env': {'GITHUB_TOKEN': '${GITHUB_TOKEN}'}}}}, indent=2), encoding='utf-8')

def ci_review_command(prompt):
    return f"claude -p {prompt!r} --output-format json --json-schema finding.schema.json"

print('files:', sorted(str(p) for p in pathlib.Path('.').rglob('*') if p.is_file()))
print(ci_review_command('Review staged changes; report only NEW issues'))

### Checkpoint 3: Verify Claude Code Team Configuration

Confirm config files exist, the rule has a glob, and no secret is committed.

In [ ]:
import re
assert pathlib.Path('CLAUDE.md').exists() and '@import' in pathlib.Path('CLAUDE.md').read_text()
assert 'paths:' in pathlib.Path('.claude/rules/testing.md').read_text()
assert 'context: fork' in pathlib.Path('.claude/skills/analyze/SKILL.md').read_text()
assert re.search(r'ghp_|sk-', pathlib.Path('.mcp.json').read_text()) is None
assert ci_review_command('x').startswith('claude -p')
print("Checkpoint 3 PASSED: team config scaffolded, no secrets committed")

## Step 4: Extraction Pipeline - Schema, Validation-Retry & Batch

**Lesson reference:** Lessons 4.3-4.5, 5.5 (Task Statements 4.3-4.5, 5.5); Scenario 6 / Exercise 3

Build the extraction pipeline: a tool schema with nullable fields and an enum+other, a validation-retry loop with error feedback that short-circuits when info is absent, a batch builder with `custom_id`, and confidence-based human-review routing.

In [ ]:
extract_invoice = {
    "name": "extract_invoice",
    "description": "Extract invoice fields; return null for absent fields.",
    "input_schema": {"type": "object", "properties": {
        "invoice_number": {"type": ["string", "null"]},
        "total": {"type": ["number", "null"]},
        "currency": {"type": "string",
            "enum": ["USD", "EUR", "GBP", "other", "unclear"]}},
        "required": ["currency"]}}

def validate(ext):
    errs = []
    items = ext.get("line_items", [])
    calc = sum(i.get("amount", 0) for i in items)
    if ext.get("total") not in (None, calc):
        errs.append(f"items sum {calc} != total {ext['total']}")
    return errs

def extract_with_retry(doc, extract_fn, max_retries=2):
    ext = extract_fn(doc, [])
    for _ in range(max_retries):
        errs = validate(ext)
        if not errs:
            return ext, []
        if any("absent" in e for e in errs):
            break  # retry won't help
        ext = extract_fn(doc, errs)
    return ext, validate(ext)

def build_batch(docs):
    return [{"custom_id": f"doc-{i}",
             "params": {"model": MODEL_HAIKU, "max_tokens": 512,
                        "messages": [{"role": "user", "content": d}]}}
            for i, d in enumerate(docs)]

def route(ext, threshold=0.7):
    return "human_review" if ext.get("confidence", 1.0) < threshold \
        or ext.get("conflict_detected") else "auto_accept"

fix_fn = lambda doc, errs: {"total": 90, "line_items": [{"amount": 60}, {"amount": 30}],
                            "currency": "USD", "confidence": 0.9}
ext, remaining = extract_with_retry("...", fix_fn)
print("extraction valid:", not remaining)
print("batch size:", len(build_batch(["a", "b", "c"])) )
print("routing:", route({"confidence": 0.5}))

### Checkpoint 4: Verify Extraction Pipeline - Schema, Validation-Retry & Batch

Confirm retry converges, batch uses custom_id, and low confidence routes to humans.

In [ ]:
ext, remaining = extract_with_retry('...', fix_fn)
assert remaining == []
assert build_batch(['x'])[0]['custom_id'] == 'doc-0'
assert route({'confidence': 0.5}) == 'human_review'
assert route({'confidence': 0.95}) == 'auto_accept'
print("Checkpoint 4 PASSED: schema + retry + batch + review routing")

## Step 5: Research Pipeline - Orchestration, Error Propagation & Provenance

**Lesson reference:** Lessons 1.2, 1.3, 5.3, 5.6 (Task Statements 1.2, 1.3, 5.3, 5.6); Scenario 3 / Exercise 4

Build the coordinator/subagent research pipeline: dynamic decomposition with a coverage check, parallel Task calls in one response, structured error propagation on timeout, and a synthesis step that preserves claim-source mappings and annotates conflicts/dates.

In [ ]:
def make_task_call(subagent, prompt):
    return {"type": "tool_use", "name": "Task",
            "input": {"subagent_type": subagent, "prompt": prompt}}

parallel_turn = [make_task_call("web_search", "AI in film"),
                 make_task_call("web_search", "AI in music")]

def subagent_error(failure_type, attempted, partial=None, alternatives=None):
    return {"ok": False, "failure_type": failure_type, "attempted_query": attempted,
            "partial_results": partial or [], "alternatives": alternatives or []}

def coordinator_recover(err):
    return {"action": (err["alternatives"] or ["proceed with partials"])[0],
            "coverage_gap": err["attempted_query"]}

def claim(text, source, published):
    return {"claim": text, "source": source, "published": published}

def synthesize(findings):
    values = {f["claim"] for f in findings}
    return {"conflict": len(values) > 1,
            "sources": [{"value": f["claim"], "source": f["source"],
                         "published": f["published"]} for f in findings]}

err = subagent_error("timeout", "AI in film", partial=["1 src"],
                     alternatives=["narrower query"])
print("parallel calls:", len(parallel_turn))
print("recovery:", coordinator_recover(err))
fs = [claim("grew 12%", "a.com", "2026-02"), claim("grew 18%", "b.com", "2025-09")]
print("synthesis:", synthesize(fs))

### Checkpoint 5: Verify Research Pipeline - Orchestration, Error Propagation & Provenance

Confirm parallelism, structured recovery, and conflict-preserving synthesis.

In [ ]:
assert len(parallel_turn) == 2  # parallel in ONE response
assert coordinator_recover(err)["coverage_gap"] == "AI in film"
out = synthesize([claim('grew 12%', 'a.com', '2026-02'),
                  claim('grew 18%', 'b.com', '2025-09')])
assert out["conflict"] and len(out["sources"]) == 2
assert all('source' in s and 'published' in s for s in out["sources"])
print("Checkpoint 5 PASSED: orchestration + error propagation + provenance")

## Verification Checklist

Confirm each item before considering the capstone complete:

- [ ] The support agent loop terminates on stop_reason, never on parsed natural language (L1.1).
- [ ] process_refund is deterministically blocked until get_customer verifies identity (L1.4).
- [ ] An interception hook blocks refunds over $500 and redirects to escalation (L1.5).
- [ ] Tools return structured errors with errorCategory and isRetryable (L2.2).
- [ ] Escalation honors explicit human requests/policy gaps and ignores sentiment (L5.2).
- [ ] CLAUDE.md uses @import and a path-scoped rule uses glob frontmatter (L3.1, L3.3).
- [ ] A skill uses context: fork with allowed-tools restrictions (L3.2).
- [ ] .mcp.json uses ${ENV} expansion and commits no literal secrets (L2.4).
- [ ] The CI command uses -p with --output-format json (L3.6).
- [ ] Extraction uses tool_use + JSON schema with nullable/enum fields (L4.3).
- [ ] Validation-retry feeds back errors and short-circuits when info is absent (L4.4).
- [ ] The batch builder correlates with custom_id and resubmits only failures (L4.5).
- [ ] Low-confidence/contradictory extractions route to human review (L5.5).
- [ ] Subagents run in parallel via multiple Task calls in one response (L1.3).
- [ ] Timeouts propagate structured error context; the workflow is not aborted (L5.3).
- [ ] Synthesis preserves claim-source mappings and annotates conflicts with dates (L5.6).

## Stretch Goals

Completed the core project? Try these extensions:

1. **Add iterative refinement to the research coordinator** — Re-delegate targeted subtopics when the synthesis coverage check finds gaps (L1.2).
2. **Add a per-file + integration review pass to the CI step** — Split a multi-file PR into local passes plus a cross-file pass and reject consensus-voting (L4.6).
3. **Add a crash-recovery manifest to the research pipeline** — Export each subagent's state and resume from a manifest after a simulated crash (L5.4).
4. **Add a fork_session comparison** — Fork a shared analysis baseline to compare two refund-handling strategies (L1.6).
5. **Calibrate the confidence threshold with a labeled set** — Tune the human-review threshold and report missed-error vs review-rate tradeoffs (L5.5).

---

## Congratulations! 🎉

You have assembled and hardened a production Claude system that spans all five CCAR-F domains and all four Preparation Exercises - applying deterministic enforcement, structured errors, calibrated escalation, schema-enforced extraction, and provenance-preserving synthesis while explicitly rejecting the exam's named anti-patterns.

You've demonstrated proficiency in:

- Designing reliable agentic loops and coordinator/subagent orchestration (Domain 1)
- Tool interface design, structured MCP errors, and MCP server integration (Domain 2)
- Claude Code configuration, path-specific rules, skills, and CI/CD integration (Domain 3)
- Structured output, validation-retry loops, and batch processing (Domain 4)
- Context preservation, escalation, error propagation, and provenance (Domain 5)

**What's next?**

- Review any lessons you found challenging
- Attempt the stretch goals above
- Book your exam when you can complete this capstone unaided